# T20 Score Prediction — Fine-tuning

## Training `meta-llama/Llama-3.2-3B` with LoRA + QLoRA

Fine-tuning the base LLaMA 3.2 3B model on our T20 first-innings score prediction dataset using supervised fine-tuning (SFT) with 4-bit quantization and LoRA adapters.

**LITE_MODE = True** → run on a free T4 GPU in Colab  
**LITE_MODE = False** → use a paid A100 (high memory) for full training

In [ ]:
!pip install -q --upgrade bitsandbytes==0.48.2 trl==0.25.1 peft accelerate transformers datasets wandb
!wget -q https://raw.githubusercontent.com/ratnayakatilanka/t20_score_prediction/main/research/util.py -O util.py

In [ ]:
import os
import re
import math
from datetime import datetime
from tqdm import tqdm
from google.colab import userdata
from huggingface_hub import login
import torch
import transformers
import wandb
from transformers import AutoModelForCausalLM, AutoTokenizer, TrainingArguments, set_seed, BitsAndBytesConfig
from peft import LoraConfig
from trl import SFTTrainer, SFTConfig
from datasets import load_dataset, Dataset, DatasetDict

In [ ]:
# ── Constants ────────────────────────────────────────────────────────────────
BASE_MODEL    = "meta-llama/Llama-3.2-3B"
PROJECT_NAME  = "t20_score_prediction"
HF_USER       = "ratnayakatilanka"
LITE_MODE     = True   # True → T4 (free), False → A100 (paid)

DATASET_NAME  = f"{HF_USER}/t20_score_prediction_prompts"

RUN_NAME      = f"{datetime.now():%Y-%m-%d_%H.%M.%S}"
if LITE_MODE:
    RUN_NAME += "-lite"
PROJECT_RUN_NAME = f"{HF_USER}/{PROJECT_NAME}-{RUN_NAME}"
FINETUNED_MODEL  = f"{HF_USER}/{PROJECT_NAME}"

# ── LoRA hyperparameters ──────────────────────────────────────────────────────
LORA_R        = 32 if not LITE_MODE else 16
LORA_ALPHA    = 64 if not LITE_MODE else 32
LORA_DROPOUT  = 0.1
TARGET_MODULES = ["q_proj", "v_proj", "k_proj", "o_proj"]

# ── Training hyperparameters ──────────────────────────────────────────────────
EPOCHS                      = 3 if not LITE_MODE else 1
BATCH_SIZE                  = 4 if not LITE_MODE else 4
GRADIENT_ACCUMULATION_STEPS = 4 if not LITE_MODE else 2
LEARNING_RATE               = 2e-4
MAX_SEQUENCE_LENGTH         = 256  # max prompt tokens = 148, completion = ~2 tokens; 256 gives safe headroom

# paged_adamw_32bit: full precision, better stability (A100)
# paged_adamw_8bit:  quantized optimizer states, saves memory (T4)
if LITE_MODE:
    OPTIMIZER = "paged_adamw_8bit"
else:
    OPTIMIZER = "paged_adamw_32bit"

# ── Quantization / precision ──────────────────────────────────────────────────
QUANT_4_BIT = True
use_bf16    = torch.cuda.is_bf16_supported()   # False on T4, True on A100

# ── Tracking ─────────────────────────────────────────────────────────────────
VAL_SIZE     = 50  if LITE_MODE else 100
LOG_STEPS    = 5   if LITE_MODE else 10
SAVE_STEPS   = 100 if LITE_MODE else 200
LOG_TO_WANDB = True

print(f"LITE_MODE           : {LITE_MODE}")
print(f"Run name            : {PROJECT_RUN_NAME}")
print(f"bfloat16            : {use_bf16}")
print(f"OPTIMIZER           : {OPTIMIZER}")
print(f"MAX_SEQUENCE_LENGTH : {MAX_SEQUENCE_LENGTH}")
print(f"VAL_SIZE: {VAL_SIZE}  |  LOG_STEPS: {LOG_STEPS}  |  SAVE_STEPS: {SAVE_STEPS}")
print(f"EPOCHS  : {EPOCHS}  |  BATCH_SIZE: {BATCH_SIZE}  |  LORA_R: {LORA_R}")

In [ ]:
# Log in to HuggingFace
hf_token = userdata.get("HF_TOKEN")
login(hf_token, add_to_git_credential=True)

## Tracking with Weights & Biases

We use [Weights & Biases](https://wandb.ai) to track training loss, validation loss, and learning rate curves in real time.

To enable tracking:
1. Sign up at https://wandb.ai and get your API key
2. In Colab, click the **key icon** in the left panel → add a secret named `WANDB_API_KEY`

Set `LOG_TO_WANDB = False` in the constants cell if you want to skip W&B logging.

In [ ]:
# Log in to Weights & Biases
if LOG_TO_WANDB:
    wandb_api_key = userdata.get("WANDB_API_KEY")
    os.environ["WANDB_API_KEY"] = wandb_api_key
    wandb.login()
    os.environ["WANDB_PROJECT"]   = PROJECT_NAME
    os.environ["WANDB_LOG_MODEL"] = "false"
    os.environ["WANDB_WATCH"]     = "false"

In [ ]:
dataset = load_dataset(DATASET_NAME)
train = dataset["train"]
val   = dataset["validation"].select(range(VAL_SIZE))
test  = dataset["test"]

print(dataset)
print(f"\nTrain : {len(train):,}")
print(f"Val   : {len(val):,}  (using {VAL_SIZE} of {len(dataset['validation']):,})")
print(f"Test  : {len(test):,}")
print("\n--- Sample ---")
print(train[0]["prompt"])
print("Completion:", train[0]["completion"])

## Load Tokenizer and Model

The model is quantized to 4 bits using `BitsAndBytesConfig` to fit within a T4 GPU (16 GB).

In [ ]:
if QUANT_4_BIT:
    quant_config = BitsAndBytesConfig(
        load_in_4bit=True,
        bnb_4bit_use_double_quant=True,
        bnb_4bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
        bnb_4bit_quant_type="nf4",
    )
else:
    quant_config = BitsAndBytesConfig(
        load_in_8bit=True,
        bnb_8bit_compute_dtype=torch.bfloat16 if use_bf16 else torch.float16,
    )

tokenizer = AutoTokenizer.from_pretrained(BASE_MODEL, trust_remote_code=True)
tokenizer.pad_token = tokenizer.eos_token
tokenizer.padding_side = "right"

base_model = AutoModelForCausalLM.from_pretrained(
    BASE_MODEL,
    quantization_config=quant_config,
    device_map="auto",
)
base_model.generation_config.pad_token_id = tokenizer.pad_token_id
base_model.eval()
print(f"Loaded: {BASE_MODEL}")

## Configure LoRA and Training Parameters

In [ ]:
# LoRA parameters
lora_parameters = LoraConfig(
    lora_alpha=LORA_ALPHA,
    lora_dropout=LORA_DROPOUT,
    r=LORA_R,
    bias="none",
    task_type="CAUSAL_LM",
    target_modules=TARGET_MODULES,
)

In [ ]:
# Training parameters
train_parameters = SFTConfig(
    output_dir=PROJECT_RUN_NAME,
    num_train_epochs=EPOCHS,
    per_device_train_batch_size=BATCH_SIZE,
    per_device_eval_batch_size=1,
    gradient_accumulation_steps=GRADIENT_ACCUMULATION_STEPS,
    optim=OPTIMIZER,
    save_steps=SAVE_STEPS,
    save_total_limit=10,
    logging_steps=LOG_STEPS,
    learning_rate=LEARNING_RATE,
    weight_decay=0.001,
    fp16=not use_bf16,
    bf16=use_bf16,
    max_grad_norm=0.3,
    max_steps=-1,
    warmup_ratio=0.03,
    lr_scheduler_type="cosine",
    eval_strategy="steps",
    eval_steps=SAVE_STEPS,
    report_to="wandb" if LOG_TO_WANDB else "none",
    run_name=RUN_NAME if LOG_TO_WANDB else None,
    dataset_text_field="prompt",
    packing=False,
    push_to_hub=True,
    hub_model_id=PROJECT_RUN_NAME,
    hub_strategy="every_save",
    hub_private_repo=True,
)

In [ ]:
fine_tuning = SFTTrainer(
    model=base_model,
    train_dataset=train,
    eval_dataset=val,
    peft_config=lora_parameters,
    args=train_parameters,
)

In [ ]:
# Fine-tune!
fine_tuning.train()

# Push fine-tuned model to Hugging Face Hub (private)
fine_tuning.model.push_to_hub(PROJECT_RUN_NAME, private=True)
print(f"Saved to the hub: {PROJECT_RUN_NAME}")

In [ ]:
if LOG_TO_WANDB:
    wandb.finish()

##Keggle link
https://www.kaggle.com/code/tilankaratnayaka/fine-tuning